In [ ]:
import pandas as pd
import numpy as np
import pyxdf
import mne
import matplotlib.pyplot as plt
from pathlib import Path
from IPython.display import display
from mne.time_frequency import stft
from scipy.signal import welch

In [ ]:
"""
In this notebook, we preprocess and ICA the raw EEG, then go through every "fired" trial
across all active runs (participant 1 run 2, participant 1 run 4, participant 2 run 1, participant 2 run 3) -- using the
same FIRED_TRIAL_INDICES as PSDprocessing.ipynb -- combine each participant's fired
trials across runs, epoch them around sonication time (-2 to 7 seconds), and compute the
time-dependent PSD and per-band power change.
"""

In [ ]:
XDF_ROOT = Path(r"C:\Users\jshin\OW_closedloopLIFU\xdf_data")
PARTICIPANTS = ["participant_1", "participant_2"]
RUNS = [1, 2, 3, 4]
CH_NAMES = ["FCz", "CP3", "P5", "Cz", "Pz", "POz", "CP4"]
DATA_COLS = ["Ch0", "Ch1", "Ch2", "Ch3", "Ch4", "Ch5", "Ch6"]  # maps 1:1 to CH_NAMES

def xdf_path_for(participant, run):
    folder = f"sub-{participant}_run_{run}"
    fname = f"sub-{participant}_run_{run}_ses-1_task-{participant}_run_{run}_run-001_eeg.xdf"
    return XDF_ROOT / folder / "ses-1" / "eeg" / fname

In [ ]:
def _load_eeg_and_lifu_markers(xdf_path):
    """Load one XDF recording; return the raw EEG dataframe (with Timestamp) and the
    array of real LIFU_ON stimulation-onset timestamps (empty array if the run has none)."""
    data, _ = pyxdf.load_xdf(str(xdf_path))
    # Streams aren't always in the same order across recordings, so look them up by name.
    streams = {s['info']['name'][0]: s for s in data}

    eeg_stream = streams['EEG_gpype']
    eeg_df = pd.DataFrame(eeg_stream['time_series'])
    eeg_df = eeg_df.rename(columns={i: f"Ch{i}" for i in range(eeg_df.shape[1])})
    eeg_df['Timestamp'] = eeg_stream['time_stamps']
    eeg_raw = eeg_df[DATA_COLS + ['Timestamp']]

    lifu_stream = streams['EEG_LIFU_events']
    lifu_df = pd.DataFrame(lifu_stream['time_series']).rename(columns={0: 'markers'})
    lifu_df['Timestamp'] = lifu_stream['time_stamps']
    lifu_on_ts = np.asarray(lifu_df.loc[lifu_df['markers'] == 'LIFU_ON', 'Timestamp'])

    return eeg_raw, lifu_on_ts


def load_clean_raw_and_lifu_events(xdf_path, sfreq=250):
    """Load one XDF recording, return a cleaned/filtered Raw plus the sample indices
    of every real LIFU_ON stimulation marker (empty array if the run has none)."""
    eeg_raw, lifu_on_ts = _load_eeg_and_lifu_markers(xdf_path)

    timestamps = eeg_raw['Timestamp'].values
    if len(lifu_on_ts) == 0:
        event_samples = np.array([], dtype=int)
    else:
        insert_idx = np.clip(np.searchsorted(timestamps, lifu_on_ts), 1, len(timestamps) - 1)
        left_vals = timestamps[insert_idx - 1]
        right_vals = timestamps[insert_idx]
        use_left = np.abs(lifu_on_ts - left_vals) < np.abs(lifu_on_ts - right_vals)
        event_samples = np.where(use_left, insert_idx - 1, insert_idx)

    # Clean EEG
    df = eeg_raw[DATA_COLS].copy()
    df = df.replace([np.inf, -np.inf], np.nan)
    df = df.replace(-200000.0, np.nan)

    # drop bad channels/datapoints
    bad_frac = df.isna().mean()
    bad_channels = list(bad_frac[bad_frac > 0.5].index)

    df = df.interpolate(method='linear', limit=5, limit_direction='both')
    df = df.bfill().ffill()
    df = df[DATA_COLS]

    # --- Build MNE Raw ---
    info = mne.create_info(ch_names=CH_NAMES, sfreq=sfreq, ch_types='eeg')
    raw = mne.io.RawArray(df.values.T, info, verbose=False)
    raw.set_montage(mne.channels.make_standard_montage("standard_1020"))
    raw.info['bads'] = [CH_NAMES[DATA_COLS.index(c)] for c in bad_channels]
    if raw.info['bads']:
        print(f"  Bad channel(s) detected: {raw.info['bads']} -> interpolating from neighbors")
        raw.interpolate_bads(reset_bads=True, verbose=False)
    # filter
    raw.notch_filter(60., verbose=False)
    raw.filter(1., 40., fir_design='firwin', verbose=False)
    raw.set_eeg_reference('average', verbose=False)

    # ICA but no exclusion
    ica = mne.preprocessing.ICA(n_components=0.99999, random_state=97, max_iter='auto', verbose=False)
    ica.fit(raw, verbose=False)

    return raw, event_samples


def make_epochs(raw, event_samples, tmin, tmax, baseline):
    if len(event_samples) == 0:
        return None
    events = np.column_stack([event_samples,
                               np.zeros_like(event_samples, dtype=int),
                               np.ones_like(event_samples, dtype=int)])
    return mne.Epochs(raw, events, event_id=1, tmin=tmin, tmax=tmax,
                       baseline=baseline, preload=True, verbose=False)

In [ ]:
# Same manually-verified firing indices as PSDprocessing.ipynb -- cross reference
# electric_artifacts.ipynb to see which trials actually sonicated.
CHECK_PRE, CHECK_POST = 2, 7  # matches new_electric_artifacts.ipynb

FIRED_TRIAL_INDICES = {
    "participant_1": {1: [], 2: [0,1,2,3,4], 3: [], 4: [1,6,7,8,9]},
    "participant_2": {1: [0,2], 2: [], 3: [0], 4: []},
}


def get_check_windows(xdf_path, fs=250):
    """Raw (uncleaned, unfiltered) per-trial EEG windows around each real LIFU_ON marker,
    plus the matched EEG sample index for each -- used to translate FIRED_TRIAL_INDICES
    (trial position within a run) into absolute sample indices."""
    eeg_raw, lifu_on_ts = _load_eeg_and_lifu_markers(xdf_path)
    timestamps = eeg_raw['Timestamp'].values
    pre_samples = int(CHECK_PRE * fs)
    post_samples = int(CHECK_POST * fs)
    window_len = pre_samples + post_samples

    windows, sample_idxs = [], []
    for event in lifu_on_ts:
        center_idx = int(np.abs(timestamps - event).argmin())
        start_idx = center_idx - pre_samples
        end_idx = start_idx + window_len
        if start_idx < 0 or end_idx > len(eeg_raw):
            continue
        windows.append(eeg_raw.iloc[start_idx:end_idx][DATA_COLS].to_numpy().T)
        sample_idxs.append(center_idx)
    return windows, np.array(sample_idxs, dtype=int)

In [ ]:
PSD_TMIN, PSD_TMAX = -2.0, 7.0
PSD_BASELINE = (-0.2, 0)

participant_fired_epochs = {}

for participant in PARTICIPANTS:
    keep_indices = FIRED_TRIAL_INDICES[participant]
    fired_samples_by_run = {}

    for run in RUNS:
        path = xdf_path_for(participant, run)
        _, sample_idxs = get_check_windows(path)
        if len(sample_idxs) == 0:
            continue
        keep = set(keep_indices.get(run, []))
        fired_samples_by_run[run] = [sample_idxs[i] for i in keep if i < len(sample_idxs)]

    fired_epoch_list = []
    for run, fired_samples in fired_samples_by_run.items():
        if not fired_samples:
            continue
        path = xdf_path_for(participant, run)
        raw, event_samples = load_clean_raw_and_lifu_events(path)
        main_epochs = make_epochs(raw, event_samples, tmin=PSD_TMIN, tmax=PSD_TMAX, baseline=PSD_BASELINE)
        keep_mask = np.isin(main_epochs.events[:, 0], fired_samples)
        if keep_mask.sum() == 0:
            print(f"  {participant} run {run}: fired trials fell too close to the recording edge -- skipping.")
            continue
        fired_epoch_list.append(main_epochs[keep_mask])

    if fired_epoch_list:
        participant_fired_epochs[participant] = mne.concatenate_epochs(fired_epoch_list, on_mismatch='warn')
        print(f"{participant}: {len(participant_fired_epochs[participant])} fired trials combined across runs")
    else:
        participant_fired_epochs[participant] = None
        print(f"{participant}: no fired trials found")

In [ ]:
for participant, epochs in participant_fired_epochs.items():
    if epochs is None or len(epochs) == 0:
        continue

    sfreq = epochs.info['sfreq']
    n_fft = 256
    step = 64

    for idx, ep in enumerate(epochs):
        Zxx = stft(ep, wsize=n_fft, tstep=step)
        power = np.abs(Zxx)**2 / (n_fft**2)  # normalizing

        times = np.linspace(epochs.tmin, epochs.tmax, power.shape[-1])
        freqs = np.linspace(0, sfreq / 2, power.shape[1])

        # frequencies you want to see (1-40Hz)
        mask = (freqs >= 1) & (freqs <= 40)
        freqs_1_40 = freqs[mask]
        power_1_40 = power[:, mask, :]

        # single graph for each epoch (averaging across channels)
        psd_avg = power_1_40.mean(axis=0)

        # convert to db for better visualizations
        psd_db = 10 * np.log10(psd_avg + 1e-20)

        # only use 95%
        vmin, vmax = np.percentile(psd_db, [5, 95])

        # plotting
        plt.figure(figsize=(10, 6))
        plt.imshow(
            psd_db,
            aspect='auto',
            origin='lower',
            extent=[times[0], times[-1], freqs_1_40[0], freqs_1_40[-1]],
            vmin=vmin,
            vmax=vmax,
            cmap='viridis'
        )
        plt.xlabel("Time (s)")
        plt.ylabel("Frequency (Hz)")
        plt.title(f"{participant} trial {idx} -- Sonicating Time-Resolved PSD (1-40 Hz, dB)")
        plt.colorbar(label="Power (dB)")
        plt.show()

In [ ]:
# PSD DATA ANALYSIS
# windowing -- pre_window is derived from each participant's epochs.tmin so it lines
# up with the actual epoch window (-2..7s) instead of a hardcoded constant

fs = 250
participant_psds = {}

for participant, epochs in participant_fired_epochs.items():
    if epochs is None or len(epochs) == 0:
        continue

    pre_window = int(abs(epochs.tmin) * fs)
    data = epochs.get_data()  # (n_trials, n_channels, n_times)

    pre_psds, post_psds = [], []
    for trial in data:
        trial = trial.T  # time x channels
        f, pre_psd = welch(trial[:pre_window], fs=fs, nperseg=fs, axis=0)
        _, post_psd = welch(trial[pre_window:], fs=fs, nperseg=fs, axis=0)
        pre_psds.append(pre_psd)
        post_psds.append(post_psd)

    participant_psds[participant] = {"freqs": f, "pre": pre_psds, "post": post_psds}

In [ ]:
def band_power(psd, freqs, f_lo, f_hi):
    idx = (freqs >= f_lo) & (freqs <= f_hi)
    return np.trapezoid(psd[idx, :], freqs[idx], axis=0)

bands = {"Theta": (4, 7), "Alpha": (8, 12), "Beta": (13, 30), "Gamma": (30, 40)}
ch_index = CH_NAMES

participant_band_results = {}

for participant, psds in participant_psds.items():
    freqs = psds["freqs"]

    # per-trial band power/percent-change/dB-change
    trial_pct = {name: [] for name in bands}
    trial_db = {name: [] for name in bands}
    for pre_psd, post_psd in zip(psds["pre"], psds["post"]):
        for name, (lo, hi) in bands.items():
            pre_bp = band_power(pre_psd, freqs, lo, hi)
            post_bp = band_power(post_psd, freqs, lo, hi)
            trial_pct[name].append((post_bp - pre_bp) / pre_bp * 100)
            trial_db[name].append(10 * np.log10(post_bp / pre_bp))

    # mean across trials
    bands_pct_df = pd.DataFrame({f"{name} %Δ": np.mean(trial_pct[name], axis=0) for name in bands}, index=ch_index)

    # std across trials -- large values here mean the mean is being driven by one
    # outlier trial rather than a consistent effect
    bands_pct_std_df = pd.DataFrame({f"{name} %Δ (trial std)": np.std(trial_pct[name], axis=0) for name in bands}, index=ch_index)

    # dB change -- same log scaling as the STFT plots, much less sensitive to a
    # noisy/small pre-window denominator than the raw percent change above
    bands_db_df = pd.DataFrame({f"{name} ΔdB": np.mean(trial_db[name], axis=0) for name in bands}, index=ch_index)

    participant_band_results[participant] = {
        "pct": bands_pct_df, "pct_std": bands_pct_std_df, "db": bands_db_df,
    }

    n_trials = len(participant_fired_epochs[participant])
    print(f"\n=== {participant} (N={n_trials} fired trials, combined across runs) ===")
    print("Percent change in band power (mean across trials):")
    display(bands_pct_df)
    print("\nStd of percent change across trials (large = unreliable / outlier-driven):")
    display(bands_pct_std_df)
    print("\ndB change in band power (log-ratio, comparable to STFT plots):")
    display(bands_db_df)